# **Credit Card Fraud Detection Exploratory Data Analysis**

## **Load the training and test data sets.**

In [1]:
import os
import pandas as pd

DATASETS_PATH = os.getenv("DATASETS_PATH")
DATASETS_PATH = DATASETS_PATH + "/AnomalyDetection/CreditCardFraudDetection/"

train_df = pd.read_csv(DATASETS_PATH + "fraudTrain.csv")

test_df = pd.read_csv(DATASETS_PATH + "fraudTest.csv")

print("Train data: ")
print(train_df)
print()
print("Test data: ")
print(test_df)


Train data: 
         Unnamed: 0 trans_date_trans_time               cc_num  \
0                 0   2019-01-01 00:00:18     2703186189652095   
1                 1   2019-01-01 00:00:44         630423337322   
2                 2   2019-01-01 00:00:51       38859492057661   
3                 3   2019-01-01 00:01:16     3534093764340240   
4                 4   2019-01-01 00:03:06      375534208663984   
...             ...                   ...                  ...   
1296670     1296670   2020-06-21 12:12:08       30263540414123   
1296671     1296671   2020-06-21 12:12:19     6011149206456997   
1296672     1296672   2020-06-21 12:12:32     3514865930894695   
1296673     1296673   2020-06-21 12:13:36     2720012583106919   
1296674     1296674   2020-06-21 12:13:37  4292902571056973207   

                                    merchant       category     amt  \
0                 fraud_Rippin, Kub and Mann       misc_net    4.97   
1            fraud_Heller, Gutmann and Zieme    groc

From the first look it seems like a synthetic or simulated data set, since real life data in finance must be anonymized. But this fact does not mean the dataset is useless, moreover it is more interesting, because one can conduct a realistic feature engineering procedure on the dataset, which is more important for a portfolio project, than if the data simulated or not.  

## **Discussion of the columns names and their meaning.**

In [2]:
print(train_df.columns)

Index(['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category',
       'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip',
       'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time',
       'merch_lat', 'merch_long', 'is_fraud'],
      dtype='str')


We have 23 columns representing time of transaction, how much money was spent, the properties of a buyer (name, gender, job, day of birth, adress, credit card number), properties of buyers location (city population and its geographic coordinates), the category of transaction, merchant properties (name, location). The goal is to create a historical profile for a customer and engineer features that describe his "normal" behaviour. Let's check first if the card number unique for each customer, or there are several cards per person (which is actually a real life scenario, since cards expire).

In [3]:
# Check if the combination of the first and last name unqiuely describes a customer
print(train_df.groupby(["first", "last"])["cc_num"].nunique().sort_values(ascending=False))

first      last    
Jennifer   Scott       2
Christine  Johnson     2
John       Nichols     2
Jeffrey    Smith       2
Justin     Bell        2
                      ..
William    Thompson    1
Willie     Jordan      1
Xavier     Beltran     1
Zachary    Allen       1
Aaron      Murray      1
Name: cc_num, Length: 973, dtype: int64


As one can see the combinations are not unique for credit card numbers, let's find out whether the person is unqiue and just have or had different credit cards.

In [4]:
# Check if the combination of the first and last have unique corresponding date of birth
print(train_df.groupby(["first", "last"])["dob"].nunique().sort_values(ascending=False))

first      last    
Jennifer   Scott       2
Christine  Johnson     2
John       Nichols     2
Jeffrey    Smith       2
Justin     Bell        2
                      ..
William    Thompson    1
Willie     Jordan      1
Xavier     Beltran     1
Zachary    Allen       1
Aaron      Murray      1
Name: dob, Length: 973, dtype: int64


In [5]:
print(train_df.groupby(["first", "last","dob"])["cc_num"].nunique().sort_values(ascending=False))

first    last      dob       
Aaron    Murray    1974-12-23    1
         Pena      1950-11-27    1
         Rogers    1945-03-15    1
         Stewart   1995-04-22    1
Adam     Keller    1932-09-17    1
                                ..
William  Thompson  1937-03-17    1
Willie   Jordan    1957-08-08    1
Xavier   Beltran   1984-06-04    1
Zachary  Allen     1969-07-24    1
         Boone     1927-12-11    1
Name: cc_num, Length: 983, dtype: int64


In [6]:
# Now the same for test data frame
print(test_df.groupby(["first", "last","dob"])["cc_num"].nunique().sort_values(ascending=False))

first    last      dob       
Aaron    Murray    1974-12-23    1
         Pena      1950-11-27    1
         Rogers    1945-03-15    1
         Stewart   1995-04-22    1
Adam     Keller    1932-09-17    1
                                ..
William  Thompson  1937-03-17    1
Willie   Jordan    1957-08-08    1
Xavier   Beltran   1984-06-04    1
Zachary  Allen     1969-07-24    1
         Boone     1927-12-11    1
Name: cc_num, Length: 924, dtype: int64


It is now clear that we have a simplified situation and a card number describes uniqely a customer.

## **Ensure that the split to train and test data sets is correct in the time scale and we prevent time-series leakage.**

In [7]:
train_df["trans_date_trans_time"] = pd.to_datetime(train_df["trans_date_trans_time"])
test_df["trans_date_trans_time"] = pd.to_datetime(test_df["trans_date_trans_time"])

train_max = train_df["trans_date_trans_time"].max()
test_min = test_df["trans_date_trans_time"].min()

print("Train data set has earliest dates then the test one: " + str(train_max < test_min))
print("Train max date: " + str(train_max))
print("Test min date: " + str(test_min))

Train data set has earliest dates then the test one: True
Train max date: 2020-06-21 12:13:37
Test min date: 2020-06-21 12:14:25


The data split is correct in time scale, so we have no time-series leakage.

## **Check how balanced the data set is (ratio of each class counts).**

In [8]:
value_counts_train = train_df["is_fraud"].value_counts()
value_counts_test = test_df["is_fraud"].value_counts()
overall_value_counts = value_counts_train + value_counts_test

print("Train Value Counts: ")
print(value_counts_train)
print("Ratio: " + str(round(value_counts_train[1] / value_counts_train[0] * 100, 2)) + " %")
print()
print("Test Value Counts: ")
print(value_counts_test)
print("Ratio: " + str(round(value_counts_test[1] / value_counts_test[0] * 100, 2)) + " %")
print()
print("Overall Value Counts: ")
print(overall_value_counts)
print("Ratio: " + str(round(overall_value_counts[1] / overall_value_counts[0] * 100, 2)) + " %")


Train Value Counts: 
is_fraud
0    1289169
1       7506
Name: count, dtype: int64
Ratio: 0.58 %

Test Value Counts: 
is_fraud
0    553574
1      2145
Name: count, dtype: int64
Ratio: 0.39 %

Overall Value Counts: 
is_fraud
0    1842743
1       9651
Name: count, dtype: int64
Ratio: 0.52 %


The data set is extremly unbalanced considering classes numbers, so that has to be taken into account during modeling.

## **Discussion of the categorical variables from the data set.**

We won't consider such categorical variables like merchant name or adress of both merchant and customer (probably not relevant for modeling), the most interesting are the product category and job title of the customers. These columns must be probably clustered in some classes in order to deliver most information, because such a job title like electrical engineer and software developer don't have to be separated and can be considered as related and generalized as engineering, reducing redundancy this way. 

In [9]:
# we start with product category:
print(train_df["category"].value_counts().sort_index())

category
entertainment      94014
food_dining        91461
gas_transport     131659
grocery_net        45452
grocery_pos       123638
health_fitness     85879
home              123115
kids_pets         113035
misc_net           63287
misc_pos           79655
personal_care      90758
shopping_net       97543
shopping_pos      116672
travel             40507
Name: count, dtype: int64


The categories seem very clustered, except for the fact that categories "*_net" (online transactions) could be more fraud prone then in-store ones, due to necessity of PIN verification. But we have to test such hypothesis for this data set. 

In [10]:
train_df["is_online"] = train_df["category"].str.contains("_net")

print(train_df.groupby("is_online")["is_fraud"].mean())

is_online
False    0.004351
True     0.013389
Name: is_fraud, dtype: float64


We see that the online purchases more fraud prone then the in-store, but if fine grained categories bring more noise then information and are overfitting prone is still not clear (there is no extreme cases, e.g. where one category has several counts, comparing to others with thousands), so let's keep it in mind for model optimizations and start with the given categories.

In [11]:
# Now the same for job title:
print(train_df["job"].value_counts().sort_index())

job
Academic librarian                      1041
Accountant, chartered                     11
Accountant, chartered certified          534
Accountant, chartered public finance    2580
Accounting technician                   4673
                                        ... 
Water engineer                          6164
Water quality scientist                  510
Web designer                            2556
Wellsite geologist                      2601
Writer                                   504
Name: count, Length: 494, dtype: int64


For the column with job titles we see high sparsity and there are high dynamic range in counts like from 11 to thousands. In this case we have to apply some grouping procedures in order to avoid overfitting and signal noise.

## **Concluding remarks for this EDA:**

* First important finding is that we have a correct split into train and test data sets relative to time scale.
* This classification problem is highly imbalanced with class counts ratio of $ \approx 0.5 \% $. This has to be taken into account during modelling.
* The categorical variable representing job title is very sparse and thus overfitting prone, so it should be recategorized.
* The credit card numbers represent customers uniquely in the dataset, so there is no case where a customer has two or more credit cards.
* There are columns with temporal and geospacial information, so one can use them for feature engineering, like e.g. distances between merchants and customers and average purchasing rate per customer.